In [1]:
!pip install faiss-cpu sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 96.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.1 MB/s eta 0:00:00:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.1 MB/s eta 0:00:0000:0100:01
  Attempting uni

In [2]:
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

2025-11-28 03:40:43.298579: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764301243.488055      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764301243.541321      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
class VietnameseRAGSystem:
    def __init__(self, embedding_model_name="intfloat/multilingual-e5-large-instruct", cross_encoder_name="namdp-ptit/ViRanker", cache_dir="/kaggle/working", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer(embedding_model_name, device=device)
        self.embedding_model.eval()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Loading cross-encoder model...")
        # Sửa: Chuyển cross-encoder sang GPU để tăng tốc
        self.cross_encoder = CrossEncoder(cross_encoder_name, device=device)

        self.index = None
        self.chunks = []
        self.metadata = []
        self.bm25_index = None
        self.normalized_texts = []
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
       
        self.retrieval_task = 'Given a query about Vietnamese history, retrieve relevant historical passages that answer the query'
   
    def normalize_text(self, text):
        return text.lower()
   
    def get_detailed_instruct(self, query):
        return f'Instruct: {self.retrieval_task}\\nQuery: {query}'
   
    def load_index(self, faiss_file, metadata_file):
        self.index = faiss.read_index(faiss_file)
       
        with open(metadata_file, 'rb') as f:
            index_data = pickle.load(f)
       
        self.chunks = index_data['chunks']
        self.metadata = index_data['metadata']
        self.normalized_texts = index_data['normalized_texts']
       
        try:
            if len(self.normalized_texts) > 0:
                tokenized_corpus = [re.sub(r'[.,!?;:\"()]+', ' ', text).split() for text in self.normalized_texts]
                self.bm25_index = BM25Okapi(tokenized_corpus)
                print("BM25 index created successfully")
            else:
                print("Empty corpus, skip BM25")
                self.bm25_index = None
        except Exception as e:
            print(f"Error creating BM25: {e}. Continue without BM25.")
            self.bm25_index = None
       
        print(f"Loaded index with {len(self.chunks)} chunks")
   
    def hybrid_search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        if self.index is None:
            raise ValueError("Index not initialized!")
       
        initial_results = self._semantic_search(
            query,
            top_k=min(50, len(self.chunks)),
            filter_trieu_dai=filter_trieu_dai,
            filter_chu_de=filter_chu_de
        )
       
        if not initial_results:
            return []
       
        reranked_results = self._rerank_with_cross_encoder(query, initial_results, top_k=top_k)
       
        return reranked_results
    
    def _semantic_search(self, query, top_k=10, filter_trieu_dai=None, filter_chu_de=None):
        instructed_query = self.get_detailed_instruct(query)
        query_embedding = self.embedding_model.encode([instructed_query], normalize_embeddings=True)
        
        candidate_size = min(top_k * 3, len(self.chunks))
        semantic_scores, semantic_indices = self.index.search(query_embedding, candidate_size)
        
        results = []
        for score, idx in zip(semantic_scores[0], semantic_indices[0]):
            if idx < len(self.chunks) and score > 0.1:
                chunk = self.chunks[idx]
                metadata = self.metadata[idx]
                
                if filter_trieu_dai and metadata.get('trieu_dai') != filter_trieu_dai:
                    continue
                if filter_chu_de and metadata.get('chu_de') != filter_chu_de:
                    continue
                
                results.append({
                    'text': chunk['text'],
                    'metadata': metadata,
                    'score': float(score),
                    'original_index': idx
                })
                
                if len(results) >= top_k:
                    break
        
        return results
   
    def _rerank_with_cross_encoder(self, query, initial_results, top_k=5):
        if not initial_results:
            return []
       
        max_text_length = 512
        documents = [result['text'][:max_text_length] for result in initial_results]
        pairs = [[query, doc] for doc in documents]
       
        print(f"Re-ranking {len(pairs)} results...")
        try:
            # Sửa: Tăng batch_size để tận dụng GPU tốt hơn
            ce_scores = self.cross_encoder.predict(pairs, batch_size=32)
        except Exception as e:
            print(f"Cross-encoder error: {e}. Use initial results.")
            return initial_results[:top_k]
       
        final_results = []
        for i, (ce_score, original_result) in enumerate(zip(ce_scores, initial_results)):
            semantic_score = original_result['score']
            combined_score = 0.7 * ce_score + 0.3 * semantic_score
           
            final_results.append({
                'text': original_result['text'],
                'metadata': original_result['metadata'],
                'score': float(combined_score),
                'ce_score': float(ce_score),
                'original_score': semantic_score,
                'reranked': True
            })
       
        final_results.sort(key=lambda x: x['score'], reverse=True)
        return final_results[:top_k]
   
    def search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        return self.hybrid_search(query, top_k, filter_trieu_dai, filter_chu_de)
   
    def get_available_filters(self):
        trieu_dais = set(m.get('trieu_dai') for m in self.metadata if m.get('trieu_dai'))
        chu_des = set(m.get('chu_de') for m in self.metadata if m.get('chu_de'))
       
        return {
            'trieu_dai': sorted(list(trieu_dais)),
            'chu_de': sorted(list(chu_des))
        }

In [4]:
rag_system = VietnameseRAGSystem()
faiss_cache_file = "/kaggle/input/vectordb/rag_index.faiss"
metadata_cache_file = "/kaggle/input/vectordb/rag_metadata.pkl"

rag_system.load_index(faiss_cache_file, metadata_cache_file)

filters = rag_system.get_available_filters()
print(f"RAG system ready with {len(rag_system.chunks)} documents")
print(f"Available dynasties: {len(filters['trieu_dai'])}")
print(f"Available topics: {len(filters['chu_de'])}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

BM25 index created successfully
Loaded index with 19509 chunks
RAG system ready with 19509 documents
Available dynasties: 1118
Available topics: 18882


In [5]:
model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
print("Model and tokenizer loaded")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model and tokenizer loaded


In [6]:
import re
import threading
import time
from transformers import TextIteratorStreamer

RETRIEVAL_THRESHOLD = 0.60

def create_concise_prompt(question, context_str):
    prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Sau đây Dữ liệu truy xuất được từ hệ thống (chỉ dùng phần dưới đây để trả lời):

{context_str}

Câu hỏi: {question}

YÊU CẦU NGHIÊM NGẶT:
- Trả lời đầy đủ, chính xác dựa trên tài liệu
- Nếu tài liệu không đủ thông tin, Hãy thừa nhận rằng không đủ thông tin và không thể trả lời
- TUYỆT ĐỐI KHÔNG thêm bất kỳ text tiếng Anh nào, không thêm markdown, không thêm code blocks.
- Trả lời xong nội dung câu hỏi thì thêm (end) rồi kết thúc.

Bắt đầu trả lời ngay bên dưới (CHỈ MỘT ĐOẠN):"""
    return prompt

def _clean_model_output(raw_text: str) -> str:
    """Làm sạch output model với xử lý end linh hoạt"""
    if not raw_text or not raw_text.strip():
        return ""
    
    text = raw_text.strip()
    
    # Tìm và cắt bỏ mọi biến thể của end (không phân biệt hoa thường)
    end_match = re.search(r'\(?end\)?', text, re.IGNORECASE)
    if end_match:
        text = text[:end_match.start()].strip()
    
    # Loại bỏ markdown và code blocks
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'`[^`]*`', '', text)
    
    # Đảm bảo không quá ngắn
    if len(text.strip()) < 10:
        return ""
    
    return text.strip()

def get_context_for_question(question, rag_system, top_k=5, threshold=RETRIEVAL_THRESHOLD):
    start_time = time.time()
    
    # Tối ưu: giảm số lượng kết quả ban đầu để tăng tốc
    results = rag_system.search(question, top_k=min(30, top_k*3))

    retrieval_time = time.time() - start_time
    print(f"  ⏱️ Thời gian retrieval: {retrieval_time:.2f}s")

    print(f"Điểm số retrieval cho câu hỏi: '{question}'")
    for i, result in enumerate(results[:3]):  # Chỉ hiển thị 3 kết quả đầu
        score = result.get('score', 0.0)
        dynasty = result.get('metadata', {}).get('trieu_dai', 'Không rõ')
        print(f"  Kết quả {i+1}: Điểm={score:.3f}, Triều đại={dynasty}")

    if not results:
        print("  ⚠️ Không có kết quả trả về từ retrieval.")
        return None

    top_score = results[0].get('score', 0.0)
    print(f"  Top score = {top_score:.3f}, threshold = {threshold:.3f}")

    if top_score < threshold:
        print("  ⚠️ Top score quá thấp -> kết luận không có đủ thông tin trong DB.")
        return None

    # Lấy top 3 kết quả và không giới hạn độ dài
    top_results = results[:min(3, len(results))]
    context_parts = []
    for i, result in enumerate(top_results, 1):
        text = result['text'].strip()
        context_parts.append(text)

    context_build_time = time.time() - start_time - retrieval_time
    print(f"  ⏱️ Thời gian xây dựng context: {context_build_time:.2f}s")
    
    return "\n\n".join(context_parts)

def ask_with_concise_rag(question, top_k=5, threshold=RETRIEVAL_THRESHOLD):
    total_start_time = time.time()
    
    print(f"🔍 Bắt đầu xử lý câu hỏi: '{question}'")
    
    # Bước 1: Retrieval
    retrieval_start = time.time()
    context = get_context_for_question(question, rag_system, top_k=top_k, threshold=threshold)
    retrieval_time = time.time() - retrieval_start

    if context is None:
        total_time = time.time() - total_start_time
        print(f"⏱️ Tổng thời gian: {total_time:.2f}s")
        return "Trả lời: Tôi không có đủ thông tin để trả lời câu hỏi này."

    # Bước 2: Generation
    generation_start = time.time()
    
    prompt = create_concise_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Tạo streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=20)
    
    # Biến để lưu kết quả và kiểm soát việc dừng
    full_output = ""
    stop_generation = False
    
    def generate_with_streamer():
        try:
            with torch.inference_mode():
                model.generate(
                    **inputs,
                    max_new_tokens=512,  # Giảm từ 300 xuống 250 để tăng tốc
                    do_sample=True,
                    temperature=0.2,     # Giảm temperature để giảm hallucination
                    top_p=0.85,          # Điều chỉnh top_p
                    repetition_penalty=1.05,  # Giảm repetition penalty
                    streamer=streamer,
                    pad_token_id=tokenizer.eos_token_id
                )
        except Exception as e:
            print(f"Generation error: {e}")
    
    # Chạy generation trong thread riêng
    generation_thread = threading.Thread(target=generate_with_streamer)
    generation_thread.start()
    
    # Streaming output
    print("🤖 Model đang trả lời: ", end="", flush=True)
    
    # Thu thập output từ streamer
    try:
        for new_text in streamer:
            if stop_generation:
                break
                
            # IN TỪNG PHẦN TEXT KHI NHẬN ĐƯỢC
            print(new_text, end="", flush=True)
            full_output += new_text
            
            # Kiểm tra nếu có bất kỳ biến thể nào của end trong output hiện tại
            # FIX: Chỉ dừng mà không in thêm "(end)"
            if re.search(r'\(?end\)?', full_output, re.IGNORECASE):
                stop_generation = True
                break
                
    except Exception as e:
        print(f"Streaming error: {e}")
    
    # Đảm bảo thread kết thúc
    generation_thread.join(timeout=5)
    
    print()  # Xuống dòng sau khi hoàn thành streaming
    
    generation_time = time.time() - generation_start
    
    # Clean model output
    clean = _clean_model_output(full_output)
    
    total_time = time.time() - total_start_time
    
    # In log thời gian chi tiết
    print(f"⏱️ Thời gian retrieval: {retrieval_time:.2f}s")
    print(f"⏱️ Thời gian generation: {generation_time:.2f}s")
    print(f"⏱️ Tổng thời gian: {total_time:.2f}s")
    
    if not clean:
        return "Trả lời: Tôi không có đủ thông tin để trả lời câu hỏi này."
    
    return f"Trả lời: {clean}"

In [7]:
# Test questions tập trung vào lịch sử Việt Nam
import time
test_questions = [
    "Sơn Tùng MTP là ai?" ,
    "Thời kỳ Đổi mới bắt đầu từ năm nào và có những thành tựu gì?",
    "Trận Điện Biên Phủ diễn ra trong thời gian nào và kết quả ra sao?",
    "Vua Quang Trung có những chiến công quan trọng nào?",
]

print("=== TEST HỆ THỐNG VỚI LOG THỜI GIAN CHI TIẾT ===\n")

for i, question in enumerate(test_questions, 1):
    print(f"{i}. Câu hỏi: {question}")
    start_time = time.time()
    answer = ask_with_concise_rag(question)
    total_time = time.time() - start_time
    print(f"{answer}")
    print(f"🕒 Tổng thời gian cho câu hỏi {i}: {total_time:.2f}s")
    print("-" * 80)
    print()

=== TEST HỆ THỐNG VỚI LOG THỜI GIAN CHI TIẾT ===

1. Câu hỏi: Sơn Tùng MTP là ai?
🔍 Bắt đầu xử lý câu hỏi: 'Sơn Tùng MTP là ai?'
Re-ranking 50 results...
  ⏱️ Thời gian retrieval: 1.78s
Điểm số retrieval cho câu hỏi: 'Sơn Tùng MTP là ai?'
  Kết quả 1: Điểm=0.254, Triều đại=Thời kỳ Pháp thuộc
  Kết quả 2: Điểm=0.254, Triều đại=Thực dân Pháp xâm lược
  Kết quả 3: Điểm=0.254, Triều đại=Triều Nguyễn
  Top score = 0.254, threshold = 0.600
  ⚠️ Top score quá thấp -> kết luận không có đủ thông tin trong DB.
⏱️ Tổng thời gian: 1.78s
Trả lời: Tôi không có đủ thông tin để trả lời câu hỏi này.
🕒 Tổng thời gian cho câu hỏi 1: 1.78s
--------------------------------------------------------------------------------

2. Câu hỏi: Thời kỳ Đổi mới bắt đầu từ năm nào và có những thành tựu gì?
🔍 Bắt đầu xử lý câu hỏi: 'Thời kỳ Đổi mới bắt đầu từ năm nào và có những thành tựu gì?'
Re-ranking 50 results...
  ⏱️ Thời gian retrieval: 0.76s
Điểm số retrieval cho câu hỏi: 'Thời kỳ Đổi mới bắt đầu từ năm nào và có